<div align="center">

# ✈ AIR TARGET IDENTIFICATION

# 1. Data Inspection

*Understanding the dataset before model development*

</div>

---

> **Objective**
>
> Before building predictive models, it is important to understand 
> the structure and reliability of the available data.
>
> This section explores the dataset from four perspectives:
> its structure, target distributions, data quality, and feature similarity.

---

In [ ]:
import os, sys, inspect, json, gc
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, lightgbm as lgb
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src import config, load, features, sampling, pipeline, train, tune, calibrate, evaluate, analysis
cfg = config.load_config('configs/scenarios.yaml', 'configs/experiment.yaml')

# QUICK = True: tiny settings, whole notebook in a few minutes
QUICK = False
# RUN_TUNING = True: rerun Optuna (hours), otherwise load results/tuning/best_params.yaml
RUN_TUNING = False

if QUICK:
    cfg["samples_per_aircraft"] = {"train": 20, "val": 5, "test": 5, "shifted_test": 5}
    cfg["lightgbm"]["n_estimators"] = 60
    cfg["lightgbm"]["early_stopping_rounds"] = 10

Path("results/metrics").mkdir(parents=True, exist_ok=True)

## Ⅰ. Dataset Structure

### 1.1 · Dataset Overview

<sub>SIZE & COMPOSITION</sub>

A preliminary overview of the dataset: the number of observations, 
available features, and unique aircraft.

In [ ]:
df = load.load_targets(cfg["data_path"])

print(f"Observations: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Aircraft: {df['name'].nunique()}")

### 1.2 · Feature Types

<sub>NUMERICAL & CATEGORICAL DATA</sub>

Examine the data types of available columns and distinguish 
numerical, categorical, and other fields.

In [ ]:
df.dtypes.value_counts()

---

## Ⅱ. Target Distribution

### 1.3 · Target Distributions

<sub>CLASS FREQUENCIES & IMBALANCE</sub>

Analyze the distribution of the three prediction targets:
aircraft name, role, and type.

The results help identify class imbalance and differences 
in class representation.

In [ ]:
display(df["name"].value_counts().describe())
display(df["role"].value_counts())
display(df["type"].value_counts())

---

## Ⅲ. Data Quality

### 1.4 · Estimated Feature Values

<sub>SPECIFICATION RELIABILITY</sub>

Measure the proportion of estimated values in each feature.

This provides an overview of how much the dataset relies 
on approximate rather than documented specifications.

In [ ]:
estimated_mask = load.estimated_mask(df)

estimated_share = (
    estimated_mask.mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .map(lambda x: f"{x:.2f}%")
)

display(estimated_share.to_frame("estimated_share"))

### 1.5 · Estimated Values Across Classes

<sub>CLASS-LEVEL COMPARISON</sub>

Compare estimated-value proportions across aircraft names, 
roles, and types.

This helps reveal whether data quality varies between classes.

In [ ]:
# role and type only: a per-aircraft table would show which cells of which aircraft are estimated
for target in ("role", "type"):
    print(f"\nEstimated share by {target}:")

    shares = (
        estimated_mask.groupby(df[target])
        .mean()
        .mul(100)
        .round(2)
    )

    shares = shares.apply(lambda col: col.map(lambda x: f"{x:.2f}%"))

    display(shares)

### 1.6 · Missing Values

<sub>DATA COMPLETENESS</sub>

Measure the proportion of missing values in each feature.

Missing values are treated separately from estimated values, 
as they represent different data-quality issues.

In [ ]:
missing_share = (
    df[load.FEATURE_COLUMNS]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .map(lambda x: f"{x:.2f}%")
)

display(missing_share.to_frame("missing_share"))

---

## Ⅳ. Feature Similarity

### 1.7 · Near-Duplicate Aircraft

<sub>REDUNDANCY ANALYSIS</sub>

Identify aircraft pairs with highly similar specifications 
using a normalized distance over numerical and categorical features.

The closest pairs are examined for potential redundancy 
in the dataset.

In [ ]:
near_duplicates = analysis.near_duplicate_pairs(df, top_n=50)

# names of near-identical pairs are a per-aircraft output: keep them local
near_duplicates.to_csv("results/metrics/per_aircraft_near_duplicates.csv", index=False)

for threshold in (0.0, 0.001, 0.01):
    print(f"pairs with distance <= {threshold}: {(near_duplicates['distance'] <= threshold).sum()}")


---

<div align="center">

*End of Section 01 · Data Inspection*

**Dataset structure and data quality have been examined.**

</div>

The inspection provides a foundation for the subsequent sampling 
and model development stages.

---



<div align="center">

# 02 — SAMPLING & DATA SPLITS

### Building reproducible training and evaluation sets

*From aircraft specifications to structured observations*

</div>

---

> **Objective**
>
> A reliable evaluation requires carefully constructed data splits.
>
> In this section, we generate separate datasets for training, 
> validation, and testing, while checking that aircraft are 
> represented consistently across the splits.
>
> **Key aspects**
> - Reproducible sampling using predefined random seeds.
> - Consistent observation counts per aircraft.
> - Separation of training, validation, and evaluation data.

---



### 2.1 · Generating Data Splits

<sub>REPRODUCIBLE SAMPLING</sub>

Generate four independent datasets using the sampling configuration:
- **Train** — model fitting.
- **Validation** — model selection and calibration.
- **Test** — final performance evaluation.
- **Shifted Test** — evaluation under a modified data distribution.

The predefined random seeds ensure reproducibility.


In [ ]:
splits = sampling.make_splits(df, cfg)

for name, split in splits.items():
    print(f"{name:15} {len(split):>8,} observations")


### 2.2 · Aircraft Representation

<sub>OBSERVATION COUNT CONSISTENCY</sub>

Verify that each aircraft contributes the expected number of observations 
to every split.

This check helps detect sampling inconsistencies and ensures that 
no aircraft is unintentionally underrepresented or overrepresented.


In [ ]:
expected_counts = {
    "train": cfg["samples_per_aircraft"]["train"],
    "val": cfg["samples_per_aircraft"]["val"],
    "test": cfg["samples_per_aircraft"]["test"],
    "shifted_test": cfg["samples_per_aircraft"]["shifted_test"],
}

for name, split in splits.items():
    counts = split["aircraft_id"].value_counts()
    expected = expected_counts[name]
    consistent = counts.eq(expected).all()

    print(f"\n{'─' * 42}")
    print(f"  {name.upper().replace('_', ' ')}")
    print(f"{'─' * 42}")
    print(f"  Aircraft:       {counts.size:>5}")
    print(f"  Observations:   {len(split):>5,}")
    print(f"  Per aircraft:   {counts.min():>3}–{counts.max():<3} (expected {expected})")
    print(f"  Consistency:    {'✓ PASS' if consistent else '✗ FAIL'}")


### 2.3 · Target Class Coverage

<sub>CLASS REPRESENTATION ACROSS SPLITS</sub>

Verify that all target classes present in the training set are also 
represented in the validation and test sets.

This check helps identify missing classes that could affect 
model evaluation and comparison.

The analysis covers all three prediction targets:
- **Aircraft name**
- **Aircraft role**
- **Aircraft type**


In [ ]:
targets = ["name", "role", "type"]

for target in targets:
    print(f"\n{'─' * 42}")
    print(f"  {target.upper()}")
    print(f"{'─' * 42}")

    reference = set(splits["train"][target].unique())

    for name, split in splits.items():
        classes = set(split[target].unique())
        missing = reference - classes

        status = "✓ PASS" if not missing else f"✗ Missing {len(missing)}"

        print(
            f"  {name:15} "
            f"{len(classes):>4} classes   {status}"
        )


### 2.4 · Class Distribution Consistency

<sub>STRATIFICATION & DISTRIBUTION SHIFT</sub>

Compare the relative frequency of each target class across data splits.

This helps detect unintended differences in class representation 
that could introduce bias into model evaluation.

The training set serves as the reference distribution.


In [ ]:
for target in ("name", "role", "type"):
    reference = splits["train"][target].value_counts(normalize=True)

    print(f"\n{'─' * 48}")
    print(f"  {target.upper()}")
    print(f"{'─' * 48}")

    for name, split in splits.items():
        distribution = split[target].value_counts(normalize=True)
        difference = (distribution.reindex(reference.index, fill_value=0) - reference).abs().max() * 100

        print(f"  {name:15} Max deviation: {difference:.4f} pp")


---

<div align="center">

*End of Section 02 · Sampling & Data Splits*

**All sampling consistency checks passed.**

</div>

---


<div align="center">

# 03 — HYPERPARAMETER TUNING

### Optimizing model configuration

*Finding effective hyperparameters through automated search*

</div>

---

> **Objective**
>
> Hyperparameter tuning aims to identify model configurations 
> that provide the best predictive performance.
>
> In this section, Optuna is used to explore the LightGBM 
> parameter space and select configurations based on 
> validation performance.
>
> **Key aspects**
> - Define and explore the hyperparameter search space.
> - Evaluate candidate configurations using validation data.
> - Apply pruning to eliminate unpromising trials.
> - Save the best configurations for subsequent training.

---



## Ⅰ. Search Configuration

### 3.1 · Tuning Configuration

<sub>SEARCH SETTINGS & EXPERIMENT DESIGN</sub>

Inspect the hyperparameter tuning configuration before starting 
the optimization process.

The configuration defines:
- Number of optimization trials and execution timeout.
- Training sample size per aircraft.
- Random seed for reproducibility.
- Pruning settings and early stopping behavior.
- Output path for the best configurations.

The search is performed separately for each target and feature scenario.


In [ ]:
from pprint import pprint

print("Tuning configuration")
pprint(cfg["tuning"])

print("\nTargets:")
pprint(cfg["targets"])

print("\nScenarios:")
for name in cfg["scenarios"]:
    columns = features.scenario_columns(cfg, name)
    print(f"  {name}: {len(columns)} features")


### 3.2 · Optimization Setup

<sub>OPTUNA STUDY INITIALIZATION</sub>

Before launching the search, inspect the optimization function 
and verify the arguments required to run the tuning process.

Each target is optimized independently under each feature scenario, 
producing a separate set of best hyperparameters.


In [ ]:
print(inspect.signature(tune.tune_all))


### 3.3 · Running Hyperparameter Optimization

<sub>OPTUNA-BASED PARAMETER SEARCH</sub>

Run independent optimization studies for each target and feature scenario.

Each study evaluates up to 30 parameter configurations using 
validation performance, while pruning helps terminate 
unpromising trials early.

The resulting best parameters will be used in the final 
model training stage.


In [ ]:
# hours on the full settings, so off by default
# scripts/run_tuning.py does the same per study and keeps finished studies if interrupted
if RUN_TUNING:
    tune.tune_all(splits=splits, spec_df=df, cfg=cfg)
else:
    print("RUN_TUNING is off: using", cfg["tuning"]["best_params_path"])


---

<div align="center">

*End of Section 03 · Hyperparameter Tuning*

**Optimal hyperparameter configurations have been identified.**

</div>

The selected configurations provide the basis for training 
the final models using the complete training dataset.

---



<div align="center">

# 04 — TRAINING & EVALUATION

### Final model development and assessment

*Training optimized models and evaluating their predictive performance*

</div>

---

> **Objective**
>
> Train the final models using the selected hyperparameters 
> and evaluate their performance on independent test datasets.
>
> **Key aspects**
> - Load the best hyperparameter configurations.
> - Train final models using the complete training data.
> - Evaluate predictive performance on the test set.
> - Assess robustness under distribution shift.
> - Compare results against baseline models.

---



## Ⅰ. Final Model Training

The hyperparameter search identified configurations with 
the best validation performance.

In this stage, the selected configurations are used to train 
the final models on the complete training dataset.


### 4.1 · Loading Best Hyperparameters

<sub>OPTIMIZATION RESULTS</sub>

Load the best hyperparameter configurations obtained during 
the Optuna optimization process.

These configurations will be used for final model training 
across all prediction targets and feature scenarios.


In [ ]:
tuned_params = tune.load_best(cfg["tuning"]["best_params_path"])

print(f"Loaded configurations: {len(tuned_params)}")

for name, params in tuned_params.items():
    print(f"  {name}: {len(params)} parameters")


### 4.2 · Training Final Models

<sub>MODEL FITTING</sub>

Train the final LightGBM models using the complete training dataset 
and the selected hyperparameters.

Each model is trained independently for a specific prediction target 
and feature scenario.

The validation dataset is used for early stopping and model selection, 
while the test datasets remain reserved for final evaluation.


In [ ]:
results, artefacts = pipeline.run(
    spec_df=df,
    cfg=cfg,
    splits=splits,
    tuned_params=tuned_params,
    model_dir="models",
    with_baselines=True,
)

# aggregate metrics only: safe to commit
results.to_csv("results/metrics/main_results.csv", index=False)

print("Training and evaluation completed.")
print(f"Result rows: {len(results)}")
print(f"Stored model artefacts: {len(artefacts)}")


## Ⅱ. Model Evaluation

Evaluate the trained models on independent datasets 
to assess their predictive performance and robustness.

The evaluation considers both standard test conditions 
and changes in the input data distribution.


### 4.3 · Test Set Evaluation

<sub>STANDARD GENERALIZATION PERFORMANCE</sub>

Evaluate the final models on the held-out test dataset.

The analysis focuses on:
- Multi-class log loss.
- Accuracy and other classification metrics.
- Performance differences across prediction targets.
- Comparison between feature scenarios.

This evaluation provides an estimate of model performance 
on previously unseen observations under the original data distribution.


In [ ]:
print("Evaluation results:")
display(results)


### 4.4 · Distribution Shift Evaluation

<sub>ROBUSTNESS UNDER CHANGED CONDITIONS</sub>

Evaluate model performance on the shifted test dataset, 
which introduces changes in the input data distribution.

The results are compared with standard test performance 
to examine how sensitive the models are to changes 
in observation conditions.

This analysis helps identify scenarios in which 
predictive performance degrades under distribution shift.


In [ ]:
shift_results = results[(results["model"] == "lightgbm")& (results["split"].isin(["test", "shifted_test"]))].copy()

shift_comparison = shift_results.pivot(
    index=["target", "scenario"],
    columns="split",
    values=["top1", "top5", "log_loss", "brier", "macro_f1"],
)

shift_comparison.columns = [f"{metric}_{split}" for metric, split in shift_comparison.columns]

shift_comparison["top1_change_pp"] = (shift_comparison["top1_shifted_test"] - shift_comparison["top1_test"]) * 100

shift_comparison["log_loss_change"] = (shift_comparison["log_loss_shifted_test"] - shift_comparison["log_loss_test"])

display(shift_comparison.round(4))


### 4.5 · Baseline Comparison

<sub>REFERENCE MODEL PERFORMANCE</sub>

Compare the trained LightGBM models against baseline approaches.

The comparison provides context for interpreting the results 
and helps determine whether the additional model complexity 
offers meaningful improvements over simpler alternatives.

Performance is examined across prediction targets 
and feature scenarios.


In [ ]:
baseline_models = [
    "lightgbm",
    "prior",
    "knn",
    "aircraft_summed",
]

baseline_results = results[(results["split"] == "test")& (results["model"].isin(baseline_models))].copy()

metrics = [
    "top1",
    "top5",
    "log_loss",
    "brier",
    "macro_f1",
]

baseline_comparison = baseline_results[["target", "scenario", "model"] + metrics].sort_values(["target", "scenario", "model"]).reset_index(drop=True)

display(baseline_comparison.round(4))


---

<div align="center">

*End of Section 04 · Training & Evaluation*

**Final models have been trained and evaluated.**

</div>

The evaluation establishes a baseline for further experiments, 
including operator filtering, ablation studies, 
and detailed error analysis.

---


<div align="center">

# 05 — OPERATOR FILTERING

### Restricting predictions to what a country actually flies

*Masking and renormalising the aircraft probabilities after calibration*

</div>

---

> **Objective**
>
> Examine how restricting the aircraft model to the types a selected country
> **operates today** changes the candidate set and the predictive performance.
>
> The filter uses the `operators` column (states that field each type now),
> not the country of origin: selecting Ukraine keeps the F-16 and the Storm Shadow,
> and dissolved states such as the USSR cannot be selected at all.
> The models never see this column, so the filter needs no retraining.
>
> **Key aspects**
> - Inspect the operator coverage of the dataset.
> - Apply the filter for the example countries.
> - Examine the resulting candidate composition.
> - Evaluate model performance before and after filtering.

---

## Ⅰ. Operator Coverage

### 5.1 · Operators in the Dataset

<sub>FILTER CONFIGURATION</sub>

Count the operator states and the types without a current operator.

Retired types have no operator, so they drop out as soon as any filter is on.
The example countries checked below are set in `filter_examples` in the experiment configuration.

In [ ]:
operators_of = dict(zip(df["name"], df["operators"]))
operator_states = pipeline.country_filter.operator_list(df["operators"])

print(f"Operator states: {len(operator_states)}")
print(f"Types with no current operator: {df['operators'].isna().sum()}")
print(f"Example countries: {cfg['filter_examples']}")

## Ⅱ. Filtered Evaluation

### 5.2 · Applying the Filter

<sub>OPERATOR-BASED SUBSETS</sub>

For each example country and scenario, score the aircraft model on the test rows of
types that country operates, with the filter off and on.

`n_classes` shows how many candidates remain after masking.

In [ ]:
country_results = []

for country in cfg["filter_examples"]:
    for scenario in cfg["scenarios"]:
        artefact = artefacts[("name", scenario)]

        result = pipeline.filter_check(artefact=artefact, spec_df=df, selected=[country])

        result["scenario"] = scenario
        country_results.append(result)

country_results = pd.concat(country_results, ignore_index=True)
country_results.to_csv("results/metrics/operator_filter_results.csv", index=False)

display(country_results)

### 5.3 · Subset Composition

<sub>CANDIDATES PER COUNTRY</sub>

Count the aircraft, roles and types each example country operates,
and how many have a different country of origin (`other_origin`).

In [ ]:
composition = []

for country in cfg["filter_examples"]:
    classes = train.class_list(df, "name")

    allowed = pipeline.country_filter.allowed_mask(classes, operators_of, [country])

    selected_aircraft = set(np.asarray(classes)[allowed])
    subset = df[df["name"].isin(selected_aircraft)]

    composition.append({
        "country": country,
        "aircraft": len(selected_aircraft),
        "other_origin": int((~subset["country"].fillna("").str.contains(country)).sum()),
        "roles": subset["role"].nunique(),
        "types": subset["type"].nunique(),
    })

display(pd.DataFrame(composition).set_index("country"))


## Ⅲ. Filter Impact

### 5.4 · Filtering Impact

<sub>BEFORE AND AFTER PROBABILITY FILTERING</sub>

Compare model performance before and after restricting 
the predicted aircraft classes to the types a selected country operates.

The comparison helps quantify the effect of country-specific 
constraints on classification performance.



In [ ]:
for country in country_results["selection"].unique():
    print(f"\n{'─' * 48}")
    print(f"  {country.upper()}")
    print(f"{'─' * 48}")

    display(country_results[country_results["selection"] == country].sort_values("scenario"))


### 5.5 · Cross-Country Comparison

<sub>COMPARATIVE PERFORMANCE ANALYSIS</sub>

Compare the effects of operator filtering 
across different countries and feature scenarios.

Consider both the number of available aircraft classes 
and the changes in predictive performance.

Interpret the results in the context of differences 
in what each country operates.


In [ ]:
metric_columns = [col for col in country_results.select_dtypes(include="number").columns if col != "n_classes"]

country_comparison = country_results.pivot(index=["selection", "scenario"], columns="filter", values=metric_columns)

country_comparison.columns = [
    f"{metric}_{filtering}"
    for metric, filtering in country_comparison.columns
]

display(country_comparison.round(4))


---

<div align="center">

*End of Section 05 · Operator Filtering*

**Operator-filtered predictions have been examined.**

</div>

The results provide additional context for understanding 
model behavior across different countries.

---



<div align="center">

# 06 — EXPERIMENTS

### Additional model investigations

*Exploring feature reliability, generalization and multi-observation performance*

</div>

---

> **Objective**
>
> Investigate model behavior under alternative experimental 
> conditions and evaluate aspects not covered by standard testing.
>
> **Key aspects**
> - Examine the contribution of estimated features.
> - Evaluate generalization to unseen aircraft families.
> - Investigate the effect of multiple observations.
> - Compare experimental results with the standard evaluation.

---



## Ⅰ. Feature Ablation

### 6.1 · Estimated Features Ablation

<sub>FEATURE RELIABILITY ANALYSIS</sub>

Evaluate the impact of estimated input features on model performance.

Train a separate set of models with estimated feature values 
removed from the input data.

Compare the resulting performance with the original models 
to determine how much the predictions depend on estimated data.


In [ ]:
ablation_results, ablation_artefacts = pipeline.run(
    spec_df=df,
    cfg=cfg,
    splits=splits,
    ablation=True,
    tuned_params=tuned_params,
    with_baselines=False,
)

ablation_results.to_csv("results/metrics/ablation_results.csv", index=False)

print("Ablation training completed.")
print(f"Result rows: {len(ablation_results)}")

display(ablation_results)


## Ⅱ. Generalization to Unseen Families

### 6.2 · Family Holdout Evaluation

<sub>GENERALIZATION BEYOND KNOWN AIRCRAFT FAMILIES</sub>

Evaluate whether the models can generalize to aircraft families 
that were excluded from the training data.

Use family-based holdout splits to ensure that aircraft 
from the same family do not appear in both training 
and evaluation subsets.

The experiment focuses on role and type classification, 
as identifying an entirely unseen aircraft by its specific 
name is not possible within the existing classification setup.

In [ ]:
family_results = []

for target in ("role", "type"):
    for scenario in cfg["scenarios"]:
        params = tuned_params.get(f"{target}_{scenario}")

        result = pipeline.unseen_families(
            spec_df=df,
            cfg=cfg,
            scenario=scenario,
            target=target,
            params=params,
        )

        family_results.append(result)

family_results = pd.concat(family_results, ignore_index=True)
family_results.to_csv("results/metrics/unseen_families.csv", index=False)

display(family_results)


## Ⅲ. Multi-Observation Evaluation

### 6.3 · Track Length Analysis

<sub>CLASSIFICATION WITH MULTIPLE OBSERVATIONS</sub>

Investigate how combining multiple observations of the same 
aircraft affects classification performance.

Compare predictions based on individual observations 
with aggregated predictions over longer observation tracks.

This experiment examines whether additional observations 
improve classification reliability and reduce prediction uncertainty.


In [ ]:
track_results = results[results["model"].astype(str).str.startswith("lightgbm_track")].copy()

track_metrics = [col for col in ("top1", "top5", "log_loss", "brier", "macro_f1") if col in track_results.columns]

display(track_results[["target", "scenario", "model", "split"] + track_metrics].sort_values(["target", "scenario", "model"]).reset_index(drop=True))


## Ⅳ. Experimental Comparison

### 6.4 · Summary of Experimental Results

<sub>COMPARATIVE ANALYSIS</sub>

Summarize the results of the additional experiments.

Compare standard and ablation models, examine performance 
across family holdout folds, and investigate the effect 
of increasing track length.

The results provide additional insight into feature reliability, 
generalization capabilities and the benefits of multiple observations.

In [ ]:
full_results = results[results["data"] == "full"].copy()

ablation_comparison = full_results.merge(
    ablation_results,
    on=["target", "scenario", "model", "split"],
    suffixes=("_full", "_ablation"),
)

comparison_metrics = [
    metric for metric in ("top1", "top5", "log_loss", "brier", "macro_f1")
    if f"{metric}_full" in ablation_comparison.columns
    and f"{metric}_ablation" in ablation_comparison.columns
]

display(
    ablation_comparison[
        ["target", "scenario", "model", "split"]
        + [
            col
            for metric in comparison_metrics
            for col in (f"{metric}_full", f"{metric}_ablation")
        ]
    ].round(4)
)


---

<div align="center">

*End of Section 06 · Experiments*

**Additional model properties have been investigated.**

</div>

The experiments provide further insight into feature reliability, 
generalization to unseen aircraft families and the effect 
of multiple observations on classification performance.

---



<div align="center">

# 07 — ERROR ANALYSIS

### Understanding model predictions and failures

*Investigating classification errors and prediction uncertainty*

</div>

---

> **Objective**
>
> Analyze model errors to identify systematic weaknesses, 
> difficult aircraft classes and patterns in incorrect predictions.
>
> **Key aspects**
> - Examine overall classification performance.
> - Identify frequently confused classes.
> - Investigate the most challenging aircraft.
> - Analyze prediction confidence and uncertainty.

---



## Ⅰ. Overall Error Overview

### 7.1 · Classification Error Summary

<sub>GENERAL PERFORMANCE OVERVIEW</sub>

Examine the overall classification performance of the trained models.

Identify the prediction targets and feature scenarios 
with the highest error rates.

This provides an initial overview of model weaknesses 
before examining individual classification errors.


In [ ]:
error_summary = []

for (target, scenario), artifact in artefacts.items():
    y_true = artifact["y"]
    proba = artifact["proba"]
    y_pred = proba.argmax(axis=1)

    error_summary.append({
        "target": target,
        "scenario": scenario,
        "samples": len(y_true),
        "correct": int((y_true == y_pred).sum()),
        "errors": int((y_true != y_pred).sum()),
        "accuracy": float((y_true == y_pred).mean()),
    })

error_summary = pd.DataFrame(error_summary)
display(error_summary.round(4))


## Ⅱ. Confusion Analysis

### 7.2 · Confusion Matrices

<sub>CLASS-LEVEL ERROR DISTRIBUTION</sub>

Analyze confusion matrices to identify classes 
that are frequently misclassified as one another.

The analysis helps reveal systematic confusion patterns 
and potential overlaps between aircraft characteristics.


In [ ]:
for (target, scenario), artifact in artefacts.items():
    if target not in ("role", "type"):
        continue

    classes = np.asarray(artifact["classes"])
    y_true = artifact["y"]
    y_pred = artifact["proba"].argmax(axis=1)

    matrix = np.zeros((len(classes), len(classes)), dtype=int)
    np.add.at(matrix, (y_true, y_pred), 1)

    print(f"\n{target.upper()} — {scenario}")

    display(pd.DataFrame(
        matrix,
        index=classes,
        columns=classes,
    ))

## Ⅲ. Difficult Classes

### 7.3 · Most Challenging Roles and Types

<sub>CLASS-SPECIFIC PERFORMANCE</sub>

Identify the roles and types with the lowest recall.

Aircraft-level recall would be a per-aircraft table, so it is not shown in the public notebook.

In [ ]:
class_performance = []

for (target, scenario), artifact in artefacts.items():
    if target == "name":
        continue

    classes = np.asarray(artifact["classes"])
    y_true = artifact["y"]
    y_pred = artifact["proba"].argmax(axis=1)

    for i, class_name in enumerate(classes):
        support = int((y_true == i).sum())
        if support == 0:
            continue

        true_positive = int(((y_true == i) & (y_pred == i)).sum())
        predicted_positive = int((y_pred == i).sum())

        class_performance.append({
            "target": target,
            "scenario": scenario,
            "class": class_name,
            "support": support,
            "recall": true_positive / support,
            "precision": (
                true_positive / predicted_positive
                if predicted_positive else 0.0
            ),
        })

class_performance = pd.DataFrame(class_performance)

for target in class_performance["target"].unique():
    print(f"\n{target.upper()} — lowest recall")
    display(class_performance[class_performance["target"] == target].sort_values("recall").head(10).round(4))

### 7.4 · Confused Aircraft Pairs

<sub>SIMILARITY AND MISCLASSIFICATION PATTERNS</sub>

Measure how concentrated the aircraft-level errors are: how many distinct pairs are confused,
and what share of all errors the 15 most frequent pairs account for.

The pair list itself is saved locally (`results/metrics/per_aircraft_confusion_pairs.csv`, gitignored).

In [ ]:
confusion_pairs = []

for (target, scenario), artifact in artefacts.items():
    if target != "name":
        continue

    classes = np.asarray(artifact["classes"])
    y_true = artifact["y"]
    y_pred = artifact["proba"].argmax(axis=1)

    n_classes = len(classes)
    counts = np.bincount(y_true * n_classes + y_pred, minlength=n_classes * n_classes).reshape(n_classes, n_classes)

    np.fill_diagonal(counts, 0)

    actual, predicted = np.nonzero(counts)

    for i, j in zip(actual, predicted):
        confusion_pairs.append({
            "scenario": scenario,
            "actual": classes[i],
            "predicted": classes[j],
            "errors": int(counts[i, j]),
        })

confusion_pairs = pd.DataFrame(confusion_pairs)
confusion_pairs.to_csv("results/metrics/per_aircraft_confusion_pairs.csv", index=False)

concentration = []
for scenario, group in confusion_pairs.groupby("scenario"):
    errors = group["errors"].sort_values(ascending=False)
    concentration.append({
        "scenario": scenario,
        "confused_pairs": len(errors),
        "errors": int(errors.sum()),
        "top15_share": float(errors.head(15).sum() / errors.sum()),
    })

display(pd.DataFrame(concentration).round(4))



## Ⅳ. Prediction Confidence

### 7.5 · Confidence and Prediction Uncertainty

<sub>PROBABILITY DISTRIBUTION ANALYSIS</sub>

Examine the confidence of correct and incorrect predictions.

Compare the probability assigned to the predicted class 
and the distribution of alternative class probabilities.

This analysis helps determine whether incorrect predictions 
are associated with low confidence or systematic overconfidence.


In [ ]:
confidence_results = []

for (target, scenario), artifact in artefacts.items():
    proba = artifact["proba"]
    y_true = artifact["y"]
    y_pred = proba.argmax(axis=1)

    confidence = proba.max(axis=1)
    correct = y_true == y_pred

    confidence_results.append({
        "target": target,
        "scenario": scenario,
        "correct_mean_confidence": (confidence[correct].mean() if correct.any() else np.nan),
        "incorrect_mean_confidence": (confidence[~correct].mean() if (~correct).any() else np.nan),
        "correct_samples": int(correct.sum()),
        "incorrect_samples": int((~correct).sum()),
    })

confidence_results = pd.DataFrame(confidence_results)
display(confidence_results.round(4))


---

<div align="center">

*End of Section 07 · Error Analysis*

**Classification errors and prediction uncertainty have been examined.**

</div>

The analysis provides insight into the limitations of the models 
and identifies areas for potential improvement.

---


 <div align="center">

# 08 — MODEL ANALYSIS & VISUALIZATION

### Evaluating model performance across different scenarios

*Comparing experimental results and exploring factors affecting prediction quality*

</div>

---

> **Objective**
>
> Analyze experimental results to evaluate model performance, compare different configurations and identify factors influencing classification quality.
>
> **Key aspects**
> - Compare model performance across scenarios and targets.
> - Evaluate calibration and prediction reliability.
> - Investigate the impact of distribution shift.
> - Analyze the effect of track length on prediction quality.
> - Examine feature importance and ablation results.

---

## Ⅰ. Performance Comparison

### 8.1 · Overall Model Performance

Compare classification performance across different targets, feature scenarios, and model configurations.

Evaluate Top-1 Accuracy, Top-5 Accuracy, and Macro-F1 to identify the strongest configurations and differences between prediction tasks.

In [ ]:
METRICS_DIR = Path("results/metrics")
FIGURES_DIR = Path("results/figures")

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(METRICS_DIR / "bundle_results.csv")

performance = results[results["split"] == "test"].copy()

performance = performance[
    ~performance["model"].str.contains(r"_track(3|5|10)$", na=False)
]

metrics = {
    "top1": "Top-1 Accuracy",
    "top5": "Top-5 Accuracy",
    "macro_f1": "Macro-F1",
}

comparison = performance[
    ["target", "scenario", "model", "top1", "top5", "macro_f1"]
].sort_values(["target", "scenario", "model"])

comparison.to_csv(
    METRICS_DIR / "performance_comparison.csv",
    index=False
)

for target in sorted(performance["target"].unique()):
    target_data = performance[performance["target"] == target]

    for metric, metric_name in metrics.items():
        pivot = target_data.pivot_table(
            index="scenario",
            columns="model",
            values=metric,
            aggfunc="mean"
        )

        ax = pivot.plot(
            kind="bar",
            figsize=(10, 5),
            width=0.8
        )

        ax.set_title(f"{metric_name} — {target.capitalize()} Classification")
        ax.set_xlabel("Feature Scenario")
        ax.set_ylabel(metric_name)
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")

        plt.xticks(rotation=0)
        plt.tight_layout()

        plt.savefig(
            FIGURES_DIR / f"{target}_{metric}_comparison.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()
        plt.close()

print("Performance comparison completed.")
print(f"Metrics saved to: {METRICS_DIR}")
print(f"Figures saved to: {FIGURES_DIR}")

## Ⅱ. Model Calibration

### 8.2 · Calibration Impact on Prediction Quality

Compare model performance before and after calibration across different targets and feature scenarios.

Examine whether calibration improves classification quality and assess its impact on prediction reliability.

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

palette = {
    "lightgbm_raw": "#AAB4C0",
    "lightgbm": "#3977B8"
}

for target in target_order:

    target_data = calibration[
        calibration["target"] == target
    ]

    for metric, metric_name in metrics.items():

        fig, ax = plt.subplots(figsize=(9, 5))

        sns.barplot(
            data=target_data,
            x="scenario",
            y=metric,
            hue="model",
            hue_order=["lightgbm_raw", "lightgbm"],
            order=scenario_order,
            palette=palette,
            errorbar=None,
            ax=ax
        )

        ax.set_title(
            f"{metric_name} — {target.capitalize()} Classification",
            fontsize=14,
            fontweight="bold",
            pad=15
        )

        ax.set_xlabel("Feature Scenario")
        ax.set_ylabel(metric_name)

        ax.grid(axis="y", alpha=0.25)
        ax.grid(axis="x", visible=False)

        ax.legend(
            title="Model",
            frameon=False
        )

        sns.despine(ax=ax)

        plt.tight_layout()

        plt.savefig(
            FIGURES_DIR / f"{target}_{metric}_calibration.png",
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()
        plt.close(fig)

## Ⅲ. Distribution Shift

### 8.3 · Performance Under Distribution Shift

Compare model performance on the original and shifted test sets.

Measure changes in classification metrics to identify configurations that are more sensitive to distribution shift and evaluate their generalization stability.

In [ ]:
METRICS_DIR = Path("results/metrics")
FIGURES_DIR = Path("results/figures")

results = pd.read_csv(METRICS_DIR / "bundle_results.csv")

metrics = {
    "top1": "Top-1 Accuracy",
    "top5": "Top-5 Accuracy",
    "macro_f1": "Macro-F1",
    "log_loss": "Log Loss"
}

shift_data = results[
    results["split"].isin(["test", "shifted_test"])
].copy()

shift_data = shift_data[
    ~shift_data["model"].str.contains(
        r"_track(3|5|10)$",
        na=False
    )
]

keys = ["target", "scenario", "model", "split", "data"]

duplicates = shift_data.duplicated(keys, keep=False)

if duplicates.any():
    raise ValueError(
        "Duplicate model results found:\n"
        + shift_data.loc[duplicates, keys].to_string(index=False)
    )

comparison = shift_data.pivot(
    index=["target", "scenario", "model", "data"],
    columns="split",
    values=list(metrics)
)

comparison.columns = [
    f"{metric}_{split}"
    for metric, split in comparison.columns
]

comparison = comparison.reset_index()

required_columns = [
    f"{metric}_{split}"
    for metric in metrics
    for split in ["test", "shifted_test"]
]

comparison = comparison.dropna(subset=required_columns)

for metric in metrics:
    comparison[f"{metric}_change"] = (
        comparison[f"{metric}_shifted_test"]
        - comparison[f"{metric}_test"]
    )

comparison.to_csv(
    METRICS_DIR / "distribution_shift_comparison.csv",
    index=False
)

sns.set_theme(style="whitegrid", context="notebook")

for metric, metric_name in metrics.items():

    fig, axes = plt.subplots(1, 3, figsize=(17, 7))

    all_changes = comparison[f"{metric}_change"].copy()

    if metric == "log_loss":
        all_changes = -all_changes

    limit = all_changes.abs().max()

    if pd.isna(limit) or limit == 0:
        limit = 0.01

    for ax, target in zip(axes, ["name", "role", "type"]):

        target_data = comparison[
            comparison["target"] == target
        ].copy()

        target_data["configuration"] = (
            target_data["scenario"]
            + " | "
            + target_data["model"]
        )

        change = target_data.pivot(
            index="configuration",
            columns="scenario",
            values=f"{metric}_change"
        )

        if metric == "log_loss":
            change = -change

        sns.heatmap(
            change,
            cmap="RdYlGn",
            center=0,
            vmin=-limit,
            vmax=limit,
            annot=True,
            fmt=".3f",
            linewidths=0.5,
            linecolor="white",
            cbar=ax == axes[-1],
            cbar_kws={"label": "Performance Improvement"},
            ax=ax
        )

        ax.set_title(
            target.capitalize(),
            fontsize=13,
            fontweight="bold",
            pad=12
        )

        ax.set_xlabel("Feature Scenario")
        ax.set_ylabel("Model Configuration")

    fig.suptitle(
        f"{metric_name} · Distribution Shift Impact",
        fontsize=16,
        fontweight="bold",
        y=1.02
    )

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR / f"distribution_shift_{metric}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

print("Distribution shift analysis completed.")

## Ⅳ. Feature Importance

### 8.4 · Feature Contribution Analysis

Examine the contribution of individual features to model predictions across different targets and feature scenarios.

Identify the most influential features and compare their relative importance between classification tasks.

In [ ]:
model_dir = Path("models")
figures_dir = Path("results/figures")
metrics_dir = Path("results/metrics")

figures_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

targets = {
    "name": "aircraft",
    "role": "role",
    "type": "type"
}

scenarios = {
    "A_radar": "A · Radar",
    "B_radar_sensors": "B · Radar + Sensors",
    "C_all": "C · All Features"
}

importance_rows = []

for target, alias in targets.items():
    for scenario in scenarios:
        model_path = model_dir / f"{alias}_{scenario}.txt"
        meta_path = model_dir / f"{alias}_{scenario}.json"

        if not model_path.exists() or not meta_path.exists():
            print(f"Missing: {model_path.name}")
            continue

        with open(meta_path, encoding="utf-8") as file:
            features = json.load(file)["features"]

        gain = [0.0] * len(features)
        split = [0] * len(features)
        pending_features = None

        with open(model_path, encoding="utf-8", errors="replace") as file:
            for line in file:
                if line.startswith("split_feature="):
                    pending_features = [
                        int(x) for x in line.partition("=")[2].split()
                    ]

                    for idx in pending_features:
                        split[idx] += 1

                elif line.startswith("split_gain=") and pending_features is not None:
                    gains = [
                        float(x) for x in line.partition("=")[2].split()
                    ]

                    if len(gains) != len(pending_features):
                        raise ValueError(
                            f"Mismatch in {model_path.name}"
                        )

                    for idx, value in zip(pending_features, gains):
                        gain[idx] += value

                    pending_features = None

        importance_rows.append(pd.DataFrame({
            "target": target,
            "scenario": scenario,
            "feature": features,
            "gain": gain,
            "split": split
        }))

        print(f"Processed: {model_path.name}")

importance_df = pd.concat(importance_rows, ignore_index=True)

importance_df["gain_share"] = (
    importance_df["gain"] /
    importance_df.groupby(["target", "scenario"])["gain"].transform("sum")
)

importance_df.to_csv(
    metrics_dir / "feature_importance.csv",
    index=False
)

sns.set_theme(style="whitegrid", context="notebook")

for target in targets:
    fig, axes = plt.subplots(1, 3, figsize=(19, 7))

    for ax, (scenario, label) in zip(axes, scenarios.items()):
        data = importance_df[
            (importance_df["target"] == target) &
            (importance_df["scenario"] == scenario)
        ]

        top = data.nlargest(15, "gain").sort_values("gain")

        ax.barh(top["feature"], top["gain"], color="#3977B8")
        ax.set_title(label)
        ax.set_xlabel("Total Gain")
        ax.set_ylabel("")

    fig.suptitle(
        f"Feature Importance · {target.title()} Classification",
        fontsize=15,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.savefig(
        figures_dir / f"feature_importance_{target}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

print("\nFeature importance analysis completed.")
print(f"Models analyzed: {importance_df.groupby(['target', 'scenario']).ngroups}")

## Ⅴ. Feature Space Visualization

### 8.5 · Aircraft Feature Space

Visualize aircraft distributions using selected physical and performance characteristics.

Examine how different aircraft types and roles occupy the feature space and identify potential overlaps between classes.

In [ ]:
from itertools import combinations

feature_labels = {
    "speed_cruise_kmh": "Cruise Speed (km/h)",
    "length_m": "Length (m)",
    "wingspan_m": "Wingspan (m)",
    "mtow_kg": "Maximum Takeoff Weight (kg)"
}

feature_cols = list(feature_labels)
pairs = list(combinations(feature_cols, 2))

sample = df[feature_cols + ["type", "role"]].copy()

sample[feature_cols] = sample[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)

sample = sample.dropna()
sample = sample[(sample[feature_cols] > 0).all(axis=1)]

sns.set_theme(style="whitegrid", context="notebook")

for target in ["type", "role"]:
    data = sample.copy()

    if target == "role":
        top_roles = data["role"].value_counts().head(5).index
        data = data[data["role"].isin(top_roles)]

    categories = sorted(data[target].unique())

    palette = dict(zip(
        categories,
        sns.color_palette("husl", len(categories))
    ))

    fig, axes = plt.subplots(2, 3, figsize=(19, 12))

    for ax, (x, y) in zip(axes.flat, pairs):
        sns.scatterplot(
            data=data,
            x=x,
            y=y,
            hue=target,
            palette=palette,
            s=40,
            alpha=0.8,
            linewidth=0,
            legend=False,
            ax=ax
        )

        ax.set_xlabel(feature_labels[x])
        ax.set_ylabel(feature_labels[y])

    handles = [
        plt.Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            color=palette[category],
            label=category,
            markersize=7
        )
        for category in categories
    ]

    fig.legend(
        handles=handles,
        title=target.title(),
        loc="center left",
        bbox_to_anchor=(0.91, 0.5)
    )

    fig.suptitle(
        f"Aircraft Feature Space · {target.title()} Classification"
        if target == "type"
        else "Aircraft Feature Space · Top 5 Roles",
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0, 0.90, 0.96])

    plt.savefig(
        f"results/figures/feature_space_{target}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"{target.title()}: {len(data)} aircraft, {len(categories)} classes")

print("\nFeature space analysis completed.")

## Ⅵ. Class Imbalance

### 8.6 · Class Distribution Analysis

Examine the distribution of classes in the training dataset for role and type classification.

Identify majority and minority classes and evaluate the degree of class imbalance that may influence classification performance.

In [ ]:
figures_dir = Path("results/figures")
metrics_dir = Path("results/metrics")

figures_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

rows = []

sns.set_theme(style="whitegrid", context="notebook")

fig, axes = plt.subplots(1, 2, figsize=(17, 8))

for ax, target in zip(axes, ["role", "type"]):
    counts = train[target].value_counts(dropna=True)

    distribution = pd.DataFrame({
        "target": target,
        "class": counts.index.astype(str),
        "count": counts.values,
        "share": counts.values / counts.sum()
    })

    rows.append(distribution)

    sns.barplot(
        data=distribution,
        x="count",
        y="class",
        color="#3977B8",
        ax=ax
    )

    ax.set_title(f"{target.title()} Class Distribution")
    ax.set_xlabel("Number of Observations")
    ax.set_ylabel("")

    ratio = counts.max() / counts.min()

    ax.text(
        0.98, 0.02,
        f"Imbalance ratio: {ratio:.1f}:1",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=10
    )

    print(f"\n{target.upper()}")
    print(f"Classes: {len(counts)}")
    print(f"Majority class: {counts.index[0]} ({counts.iloc[0]:,})")
    print(f"Minority class: {counts.index[-1]} ({counts.iloc[-1]:,})")
    print(f"Imbalance ratio: {ratio:.2f}:1")

plt.suptitle(
    "Training Data · Class Imbalance Analysis",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    figures_dir / "class_imbalance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

imbalance_df = pd.concat(rows, ignore_index=True)

imbalance_df.to_csv(
    metrics_dir / "class_distribution.csv",
    index=False
)

print("\nClass imbalance analysis completed.")

---

<div align="center">

*End of Section 08 · Model Analysis & Visualization*

**Model performance and experimental results have been evaluated.**

</div>

The analysis highlights the strengths and limitations of the models 
and identifies factors influencing classification performance.

---

## 9. Conclusions

TODO: summarize the main experimental findings, compare model performance, discuss limitations and identify possible directions for future improvements.